<div style="width: 30em; float: right; padding: 3em; border: 5px red solid; background-color: darkred; color: white"><p style="font-size: large; font-weight:bold">Rename this notebook before running any cells!</p><ol><li>Right-click on the tab title above or on the notebook in the file-browser on the left and select rename.</li><li>Remove the "_orig" part of the file name.</li></ol><p style="font-size: small">Your edits to a file ending in <code>_orig</code> may be overwritten the next time the course materials are updated.</p></div>

# Deep Dive: Why Functions Are Designed This Way in Python

**IFI8410 &mdash; Session 4: Functions and Decomposition**

Functions are the main way Python turns a program from a sequence of commands into a
system of small, understandable behaviors. For data scientists, they are not merely a
way to avoid duplicated code: they create **contracts** that make analysis reproducible,
testable, auditable, and safer to run at scale.

This notebook is a *deep dive*. Every code cell is runnable &mdash; run them, then break
them on purpose. The cells marked **Try it** are for you to edit.

### What you will work through

| Section | Question it answers |
|---|---|
| 1. The problem programmers face | Why is a long script hard to trust? |
| 2. Functions as contracts | What promise does a function make? |
| 3. Return versus print | Why does my function return `None`? |
| 4. Scope and reuse | Why can't the caller see my local variables? |
| 5. Decomposition is design | How do I split a task into functions? |
| 6. Types, edge cases, and safety | What should happen for bad input? |
| 7. Side effects and mutation | Did my function change my data? |
| 8. Parallel execution | Why does shared state break under concurrency? |
| 9. Docstrings and tests | How do I prove the contract holds? |

---

## 1. The Problem Programmers Face

As programs grow, a notebook or script can become difficult to trust. A calculation may
work once on a clean dataset but fail on missing values, silently mutate the table you
passed in,
depend on a global setting left over from an earlier cell, or return a different result
when executed concurrently.

These are not minor style issues. They affect whether another person &mdash; or you next
week &mdash; can answer basic questions:

- What is this code supposed to do?
- What data does it require?
- What does it produce?
- Which assumptions does it make?
- What happens for invalid, missing, or extreme inputs?
- Does it change anything outside itself?
- Can I test it independently of the rest of the pipeline?
- Can I safely execute it many times, or in parallel?

A function is Python's primary mechanism for putting a boundary around those questions.

> A function is a named unit of behavior with an explicit contract: inputs, result,
> assumptions, and responsibility.

For example, a function named `mean_positive_age` should have one focused
responsibility: accept a collection of ages, validate them, and return their mean. It
should not also print a report, write a CSV, alter a global counter, or modify the input
data unless its name and documentation explicitly promise those behaviors.

### A notebook that has become hard to trust

The cell below is realistic "grown-organically" notebook code. Read it and try to answer
the eight questions above about it.

In [ ]:
# Untrustworthy notebook code: everything is tangled together.

ages = [34, 28, -5, 41, None, 19]
threshold = 18          # set in some earlier cell... which one?
running_total = 0       # accumulates across re-runs!

for age in ages:
    if age is not None and age >= threshold:
        running_total += age
        print("counted", age)

print("total:", running_total)

Problems with the cell above:

1. **Re-running changes the answer.** `running_total` accumulates only because the cell
   also re-initializes it &mdash; move that line and re-running silently doubles the result.
2. **The `-5` is silently dropped** by the `>= threshold` test. Was that intended, or an
   accident?
3. **`threshold` is invisible state.** Its value comes from somewhere else in the notebook.
4. **There is no result to reuse.** The answer went to the screen, not to a variable a
   later step or a test can use.
5. **Nothing here can be tested** without re-running the whole cell.

**Try it.** Run the cell above twice *without* re-running the `running_total = 0` line
(copy just the loop and the prints into the cell below). What happens to the total?

In [ ]:
### Try it: copy ONLY the loop and the print statements below, then run this cell twice.
### Predict the second total before you run it.

### Enter your code here ###

---

## 2. Functions as Contracts

A function's contract has four parts:

| Contract element | Question it answers | Example |
|---|---|---|
| **Name** | What is its one job? | `calculate_discounted_total` |
| **Parameters** | What inputs does it accept? | `prices: list[float]`, `discount: float` |
| **Return value** | What result does it provide? | A single `float` total |
| **Preconditions and edge cases** | What must be true, and what happens if it is not? | Prices must be non-negative; discount must be between 0 and 1 |

Here is that contract written out in full. Notice how much of the function body is
*checking the contract* rather than doing arithmetic &mdash; that is normal for code at a
boundary.

In [ ]:
def calculate_discounted_total(
    prices: list[float],
    discount: float,
) -> float:
    """Return the total price after applying one fractional discount.

    Args:
        prices: Non-negative item prices.
        discount: A fraction from 0.0 through 1.0.

    Returns:
        The sum of prices after the discount is applied.

    Raises:
        TypeError: If prices or discount have inappropriate types.
        ValueError: If a price is negative or discount is outside [0, 1].
    """
    if not isinstance(discount, (int, float)):
        raise TypeError("discount must be a number")

    if not 0.0 <= discount <= 1.0:
        raise ValueError("discount must be between 0.0 and 1.0")

    if any(not isinstance(price, (int, float)) for price in prices):
        raise TypeError("every price must be a number")

    if any(price < 0 for price in prices):
        raise ValueError("prices cannot be negative")

    return sum(prices) * (1 - discount)

A caller can use the function without needing to know its internal implementation:

In [ ]:
total = calculate_discounted_total([12.50, 7.25, 4.00], 0.10)
print(total)

That separation is the point. The caller relies on the contract; the function author is
free to improve the implementation as long as the contract remains true.

Now watch the contract *defend itself*. Each call below violates one clause.

In [ ]:
# Each of these violates the contract in a different way.
# Note WHICH exception type is raised and why.

for bad_call in [
    lambda: calculate_discounted_total([10.0], 1.5),      # discount out of range
    lambda: calculate_discounted_total([10.0], "10%"),    # discount wrong type
    lambda: calculate_discounted_total([10.0, -2.0], 0.1),  # negative price
    lambda: calculate_discounted_total([10.0, "7.25"], 0.1),  # price wrong type
]:
    try:
        bad_call()
    except (TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")

Compare that to what happens **without** validation: the failure is silent and the number
looks plausible.

In [ ]:
def discounted_total_unchecked(prices, discount):
    return sum(prices) * (1 - discount)

# A 150% discount produces negative revenue -- and nobody is told.
print(discounted_total_unchecked([12.50, 7.25, 4.00], 1.5))

### Type annotations

Python supports contract communication with parameter annotations and return
annotations:

```python
def normalize(values: list[float]) -> list[float]:
    ...
```

These annotations improve readability and enable tools such as linters, IDEs, and static
type checkers to detect mismatches. They are **not normally enforced by the Python
runtime**, so type hints should be paired with validation where bad input would create a
misleading result, obscure failure, or costly downstream error.

The cell below proves the "not enforced" part.

In [ ]:
def normalize(values: list[float]) -> list[float]:
    """Scale values so the largest becomes 1.0."""
    largest = max(values)
    return [value / largest for value in values]

# The annotation says list[float]. Python does not check it at runtime.
print(normalize.__annotations__)
print(normalize([2, 4, 8]))       # ints, not floats -- runs fine
print(normalize.__doc__)

In [ ]:
### Try it: write a contract for a function `apply_late_fee(balance, days_late)`.
### - balance must be a non-negative number
### - days_late must be a non-negative integer
### - the fee is 1.5% of the balance per day late
### Raise TypeError for wrong types, ValueError for wrong values, and return the new balance.

### Enter your code here ###

---

## 3. Return Versus Print

One of the most important distinctions for beginners is that `print()` **displays** a
value, while `return` **gives a value back** to the calling code.

In [ ]:
def print_square(number: int) -> None:
    print(number * number)

result = print_square(5)
print(result)

The output is:

```text
25
None
```

Why? `print_square` has no `return` statement, so Python returns `None` implicitly. The
value was shown to a *person* but was not supplied to the *program* for further
computation.

A computational function should usually return its answer:

In [ ]:
def square(number: int) -> int:
    return number * number

result = square(5)
print(result)          # 25
print(square(5) + 3)   # 28

The difference becomes concrete the moment you try to *compose* the two.

In [ ]:
# Returning composes. Printing does not.
print(square(square(3)))          # 81

try:
    print_square(print_square(3))  # TypeError: None * None
except TypeError as error:
    print(f"TypeError: {error}")

This distinction matters immediately in automated testing. A test can compare a returned
value reliably:

In [ ]:
def test_square():
    assert square(5) == 25
    assert square(-3) == 9

test_square()
print("test_square passed")

Testing printed output is possible, but it is usually the wrong abstraction for ordinary
data transformation or calculation. Printing is a **side effect** intended for display;
returning is an **interface** intended for composition.

### A practical rule

- **Return** data that another function, test, notebook cell, or pipeline step might need.
- **Print** only when communicating with an interactive human user.
- If a function must both compute and report, **separate those jobs**: one function
  returns data, another formats or prints it.

Here is that separation applied.

In [ ]:
def summarize_scores(scores: list[float]) -> dict[str, float]:
    """Return summary statistics for scores. Computes only -- no output."""
    return {
        "count": len(scores),
        "lowest": min(scores),
        "highest": max(scores),
        "mean": sum(scores) / len(scores),
    }


def format_summary(summary: dict[str, float]) -> str:
    """Return a human-readable one-line report. Formats only -- no computation."""
    return (
        f"n={summary['count']:.0f}  "
        f"range={summary['lowest']:.1f}-{summary['highest']:.1f}  "
        f"mean={summary['mean']:.2f}"
    )


scores = [88.0, 92.5, 79.0, 95.5, 84.0]
summary = summarize_scores(scores)

print(format_summary(summary))   # for the human
print(summary["mean"])           # for the program

In [ ]:
### Try it: `report_total` below prints instead of returning, so it cannot be tested.
### Rewrite it as two functions: one that returns the total, one that formats it.

def report_total(prices):
    print(f"Total: ${sum(prices):.2f}")

### Enter your code here ###

---

## 4. Scope and Reuse

Python creates a **local scope** for every function call. Names created inside a function
belong to that call and ordinarily disappear when the function returns.

In [ ]:
def convert_celsius_to_fahrenheit(celsius: float) -> float:
    multiplier = 9 / 5
    fahrenheit = celsius * multiplier + 32
    return fahrenheit

temperature_a = convert_celsius_to_fahrenheit(0)
temperature_b = convert_celsius_to_fahrenheit(25)
print(temperature_a, temperature_b)

The names `multiplier` and `fahrenheit` are **local**. Other code cannot accidentally use
or overwrite them. This isolation makes functions safe to reuse.

In [ ]:
# The locals do not leak into the notebook's namespace.
try:
    print(multiplier)
except NameError as error:
    print(f"NameError: {error}")

Each call has its own local values. The second call does not inherit temporary state from
the first &mdash; you can see the separate frames if you print from inside.

In [ ]:
def convert_verbose(celsius: float) -> float:
    multiplier = 9 / 5
    fahrenheit = celsius * multiplier + 32
    print(f"  inside this call: celsius={celsius}, fahrenheit={fahrenheit}")
    return fahrenheit

print("call 1:"); convert_verbose(0)
print("call 2:"); convert_verbose(25)
print("call 3:"); convert_verbose(100)

### Global state creates hidden dependencies

By contrast, a function that reads a global looks simpler than it is:

In [ ]:
tax_rate = 0.08

def total_with_tax_global(price: float) -> float:
    return price * (1 + tax_rate)   # depends on something not in the signature

print(total_with_tax_global(100.0))

The function looks like it accepts only `price`, but it actually depends on `tax_rate`,
which could be changed anywhere in the program.

In [ ]:
tax_rate = 0.10          # changed somewhere else entirely -- maybe cells below!

print(total_with_tax_global(100.0))   # same call site, different answer

Now the same call can produce a different result without any change at the call site.
That makes the function harder to test, reproduce, reuse, and audit.

Prefer an **explicit dependency**:

In [ ]:
def total_with_tax(price: float, tax_rate: float) -> float:
    """Return price plus tax. Both inputs are visible at the call site."""
    return price * (1 + tax_rate)

print(total_with_tax(100.0, 0.08))
print(total_with_tax(100.0, 0.10))   # the difference is now in the record

Now the caller sees all inputs that influence the result.

Globals sometimes have legitimate uses &mdash; constants, configuration objects, logging, or
carefully managed application state &mdash; but they should be **deliberate and visible**
rather than accidental. For analytical code, explicit inputs are especially valuable
because they preserve the *provenance* of a result: the call itself records what produced
the number.

### `global` makes the dependency worse, not better

In [ ]:
counter = 0

def bump() -> None:
    global counter        # this function now reaches out and edits the notebook
    counter += 1

bump(); bump(); bump()
print(counter)

# Re-run this cell: the answer changes every time. That is not reproducible.

In [ ]:
### Try it: `apply_curve` below silently depends on a global. Rewrite it so that every
### input appears in the signature, then call it twice with different curves.

curve_points = 5

def apply_curve(score):
    return score + curve_points

### Enter your code here ###

---

## 5. Decomposition Is Design

Function decomposition should happen **while designing** a solution, not after a large
block of code has become difficult to read.

Suppose the larger task is:

> Load survey data, clean age values, calculate grouped statistics, and export a report.

Instead of one long function, identify behaviors that can be named and independently
checked. The skeleton below uses `...` bodies &mdash; naming the stages *is* the design step.

In [ ]:
# The design, before any implementation exists.
# A "table" here is the Session 3 structure: a list of dictionaries,
# one dictionary per row, dictionary keys as column names.

def load_survey_data(path: str) -> list[dict]:
    ...

def validate_age(age: object) -> "int | None":
    ...

def clean_age_column(rows: list[dict]) -> list[dict]:
    ...

def summarize_by_region(rows: list[dict]) -> list[dict]:
    ...

def export_summary(summary: list[dict], path: str) -> None:
    ...

print("Five named stages. Each one can now be discussed, tested, and replaced.")

Each function should correspond to a meaningful question:

- Does `validate_age` reject invalid strings, missing values, and implausible negative ages?
- Does `clean_age_column` leave the original rows unchanged, or does its contract
  explicitly say it mutates it?
- Does `summarize_by_region` return a table with known columns and types?
- Does `export_summary` perform **only** output, not data cleaning or statistical calculation?

This creates a pipeline whose stages can be tested independently. It also lets you replace
a stage &mdash; reading from a database instead of a CSV file, or swapping today's list of
dictionaries for a pandas `DataFrame` in Session 8 &mdash; without rewriting every other
stage. Only `load_survey_data` changes; `summarize_by_region` never learns where the rows
came from.

### The dataset

The table is the Session 3 structure: a **list of dictionaries**, where each dictionary is
one **row** and each key is a **column name**. Read it before you read the code that uses
it &mdash; it is small enough to check the program's answer with your own eyes.

The `age` column arrives as text, the way a CSV file would deliver it, and three of the
seven rows are unusable in a different way.

In [ ]:
# One dictionary per survey response. The outer list keeps the rows in order;
# each inner dictionary maps a column name to that row's value.

survey = [
    {"respondent": "r1", "region": "North", "age": "34"},
    {"respondent": "r2", "region": "North", "age": "28"},
    {"respondent": "r3", "region": "South", "age": ""},          # missing
    {"respondent": "r4", "region": "South", "age": "-5"},        # implausible
    {"respondent": "r5", "region": "South", "age": "41"},
    {"respondent": "r6", "region": "North", "age": "not sure"},  # unparseable
    {"respondent": "r7", "region": "East",  "age": "19"},
]

print("rows:", len(survey))
print("columns:", list(survey[0]))
print("first row:", survey[0])

In [ ]:
def validate_age(age: object) -> "int | None":
    """Return age as an int, or None when it is missing or implausible.

    Args:
        age: A raw age value from the survey, typically a string.

    Returns:
        An int between 0 and 120 inclusive, or None if the value cannot be used.
    """
    try:
        parsed = int(str(age).strip())
    except (TypeError, ValueError):
        return None

    if not 0 <= parsed <= 120:
        return None

    return parsed


def clean_age_column(rows: list[dict]) -> list[dict]:
    """Return NEW rows with a validated integer 'age'. Does not modify rows."""
    cleaned = []
    for row in rows:
        age = validate_age(row["age"])
        if age is not None:
            cleaned.append({**row, "age": age})
    return cleaned


def summarize_by_region(rows: list[dict]) -> list[dict]:
    """Return one summary record per region, sorted by region name."""
    by_region: dict[str, list[int]] = {}
    for row in rows:
        by_region.setdefault(row["region"], []).append(row["age"])

    return [
        {"region": region, "n": len(ages), "mean_age": sum(ages) / len(ages)}
        for region, ages in sorted(by_region.items())
    ]


def format_summary(summary: list[dict]) -> str:
    """Return a printable table. Formatting only."""
    lines = [f"{'region':<8}{'n':>4}{'mean_age':>10}"]
    for record in summary:
        lines.append(f"{record['region']:<8}{record['n']:>4}{record['mean_age']:>10.1f}")
    return "\n".join(lines)

In [ ]:
# The pipeline reads as its own documentation.
cleaned = clean_age_column(survey)
summary = summarize_by_region(cleaned)

print(format_summary(summary))
print()
print("dropped rows:", len(survey) - len(cleaned))
print("original data untouched:", survey[2])

Each stage can now be checked **on its own**, without loading a file or running the whole
pipeline:

In [ ]:
# validate_age is testable in isolation -- one line per policy decision.
assert validate_age("34") == 34
assert validate_age(" 41 ") == 41      # whitespace tolerated
assert validate_age("") is None        # missing
assert validate_age("not sure") is None  # unparseable
assert validate_age("-5") is None      # implausible
assert validate_age("200") is None     # implausible
assert validate_age(None) is None

print("validate_age: all policy cases hold")

### What makes a good function

A good function is usually:

- **Named with a verb and meaningful object**: `filter_valid_rows`, `compute_rmse`, `parse_timestamp`.
- **Focused on one responsibility.**
- **Small enough to understand in one reading**, though not artificially short.
- **Explicit about inputs and outputs.**
- **Conservative about parameters**; avoid a vague collection of positional arguments.
- **Predictable about return types.**
- **Free of hidden mutation and unrelated I/O**, unless that behavior is its stated purpose.

In [ ]:
### Try it: decompose this into named functions before you change any logic.
### Suggested stages: parse_reading -> drop_outliers -> daily_mean -> format_report

readings = ["20.1", "21.5", "x", "19.8", "999", "22.0", ""]

total = 0
n = 0
for r in readings:
    try:
        v = float(r)
    except ValueError:
        continue
    if v > 100:
        continue
    total += v
    n += 1
print("mean:", total / n)

### Enter your code here ###

---

## 6. Types, Edge Cases, and Safety

Python is **dynamically typed**: a name can refer to values of different types at runtime.
That flexibility is useful for exploration, but it means a function author must decide
what invalid input *means*.

Consider division:

```python
def average(total: float, count: int) -> float:
    return total / count
```

What should happen when:

| Input | Question |
|---|---|
| `total` is `"25"` rather than `25` | Should a numeric string be accepted, coerced, or rejected? |
| `count` is `0` | Division by zero &mdash; error, or `nan`? |
| `count` is `None` | A missing count &mdash; error, or skip? |
| `count` is `-4` | Negative observations are meaningless. |
| `count` is `3.5` | A fractional number of observations? |
| `total` is `float("nan")` | Propagate the `nan`, or refuse? |

Run the naive version against those inputs and see what Python does by default.

In [ ]:
def average_naive(total, count):
    return total / count

for total, count in [(25, 5), ("25", 5), (25, 0), (25, None), (25, -4), (25, 3.5),
                     (float("nan"), 5)]:
    try:
        print(f"average_naive({total!r}, {count!r}) = {average_naive(total, count)!r}")
    except Exception as error:
        print(f"average_naive({total!r}, {count!r}) -> {type(error).__name__}: {error}")

Note the two *dangerous* rows: `(25, -4)` returns `-6.25` and `(nan, 5)` returns `nan`.
Neither raises. Both would flow downstream into a report.

The right answer depends on the domain, but the function should make its answer
**intentional**. For a count of observations, zero and negative values are likely invalid:

In [ ]:
def average(total: float, count: int) -> float:
    """Return the arithmetic mean for a positive integer observation count."""
    if not isinstance(total, (int, float)):
        raise TypeError("total must be numeric")

    if isinstance(count, bool) or not isinstance(count, int):
        raise TypeError("count must be an integer")

    if count <= 0:
        raise ValueError("count must be greater than zero")

    return total / count


print(average(12.0, 3))

Two details in that code are worth pausing on:

- `isinstance(count, bool)` is checked **first** because in Python `bool` is a subclass of
  `int`, so `isinstance(True, int)` is `True`. Without that guard, `average(12.0, True)`
  would quietly compute `12.0`.
- The checks run **before** any arithmetic, so the function fails early and cheaply.

In [ ]:
# Why the bool guard exists:
print("isinstance(True, int) ->", isinstance(True, int))
print("True + True ->", True + True)

for total, count in [(12.0, 0), (12.0, -4), ("12", 3), (12.0, 3.5), (12.0, True)]:
    try:
        print(f"average({total!r}, {count!r}) = {average(total, count)}")
    except (TypeError, ValueError) as error:
        print(f"average({total!r}, {count!r}) -> {type(error).__name__}: {error}")

### `TypeError` versus `ValueError`

The distinction is useful:

- Raise **`TypeError`** when the *kind of object* is inappropriate: passing a string where
  a number is required.
- Raise **`ValueError`** when the *type is acceptable* but its value violates the contract:
  a negative count or a zero denominator.

Callers can then respond differently to the two: a `TypeError` usually means a bug in the
calling code, while a `ValueError` often means bad data.

### Raise, or return a missing value?

For data science, an alternative is sometimes more appropriate: instead of raising an
exception for bad rows, **return a missing value and record why**. The choice should
reflect the data-quality policy.

The key point is not that every function must reject every unexpected case &mdash; it is that
the behavior must be **explicit, consistent, and testable**.

In [ ]:
def average_or_none(total: object, count: object) -> "tuple[float | None, str | None]":
    """Return (mean, reason_rejected). One of the two is always None.

    Use this style at a row-level boundary where one bad record should not
    stop the whole job, and the reason must be recorded for the data-quality log.
    """
    if not isinstance(total, (int, float)) or isinstance(total, bool):
        return None, "total is not numeric"
    if isinstance(count, bool) or not isinstance(count, int):
        return None, "count is not an integer"
    if count <= 0:
        return None, "count must be greater than zero"
    return total / count, None


rejects = []
for total, count in [(12.0, 3), (12.0, 0), ("12", 3), (30.0, 4)]:
    mean, reason = average_or_none(total, count)
    if mean is None:
        rejects.append(((total, count), reason))
    else:
        print(f"mean = {mean}")

print("rejected:", rejects)

### Where to put runtime checks

Type hints clarify the expected interface, but they are documentation and tool-facing
metadata **by default, not runtime enforcement**. Static checkers (mypy, pyright) can flag
a likely mismatch *before* execution.

Explicit runtime checks are worth their cost at **boundaries**:

| Boundary | Why check |
|---|---|
| File ingestion | The file's contents are outside your control. |
| Web APIs / user input | Untrusted and often malformed. |
| Numerical kernels | Must fail early and clearly rather than emit `nan`. |

Inside a tight internal loop between two functions you wrote and tested, repeated
validation is usually noise.

In [ ]:
### Try it: decide and implement an edge-case policy for `percent_change(old, new)`.
### Consider: old == 0, old negative, either value non-numeric, either value nan.
### Document your choices in the docstring, then demonstrate each one.

### Enter your code here ###

---

## 7. Side Effects and Mutation

A function has a **side effect** when it changes something beyond producing its return
value. Common examples include:

- Printing output.
- Writing a file or database record.
- Sending a network request.
- Changing a global variable.
- Mutating a passed-in list or dictionary &mdash; including a row of a table.
- Changing an object's attributes.

Side effects are not automatically bad. Saving a model *must* write something; an `append`
operation is *intended* to change a collection. The danger is **unannounced** side effects.

In [ ]:
def remove_missing(values: list) -> list:
    values[:] = [value for value in values if value is not None]   # mutates the caller's list
    return values


measurements = [2.1, None, 3.4]
cleaned = remove_missing(measurements)

print("returned:", cleaned)
print("original:", measurements)     # [2.1, 3.4] -- the raw data is gone
print("same object?", cleaned is measurements)

The function appears to return a cleaned list, but it also modifies the caller's original
list. This may surprise a caller who expects `measurements` to remain available for audit
or comparison &mdash; and here the raw data is now **unrecoverable**.

A safer default is to **return a new object**:

In [ ]:
def without_missing(values: list) -> list:
    """Return a NEW list with None values removed. Input is unchanged."""
    return [value for value in values if value is not None]


measurements = [2.1, None, 3.4]
cleaned = without_missing(measurements)

print("returned:", cleaned)
print("original:", measurements)     # [2.1, None, 3.4] -- still auditable
print("same object?", cleaned is measurements)

If mutation is intentionally valuable for performance or semantics, **communicate it** in
the name, the documentation, and the return convention. Returning `None` is the Python
convention for "I mutated in place" &mdash; it is why `list.sort()` returns `None` while
`sorted()` returns a new list.

In [ ]:
def remove_missing_in_place(values: list) -> None:
    """Remove None values from values by mutating the supplied list.

    Returns:
        None. The list is modified in place, following the convention of
        list.sort() and list.reverse().
    """
    values[:] = [value for value in values if value is not None]


measurements = [2.1, None, 3.4]
remove_missing_in_place(measurements)
print(measurements)

# The same convention in the standard library:
numbers = [3, 1, 2]
print("numbers.sort() returns:", numbers.sort(), "-> numbers is now", numbers)
print("sorted([3, 1, 2]) returns:", sorted([3, 1, 2]))

### The classic trap: a mutable default argument

A default value is evaluated **once**, when the `def` statement runs &mdash; not on each call.
A mutable default is therefore shared by every call that omits the argument.

In [ ]:
def collect_broken(item, bucket=[]):       # DO NOT DO THIS
    bucket.append(item)
    return bucket

print(collect_broken("a"))
print(collect_broken("b"))    # ['a', 'b'] -- the default kept the previous call's data
print(collect_broken("c"))


def collect(item, bucket=None):            # the correct idiom
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(collect("a"))
print(collect("b"))           # ['b'] -- a fresh list each time

### For a whole table, make the choice visible

The same discipline applies when the input is a table &mdash; a list of dictionaries like
the coffee-cart data from Session 3. **Copy first, then modify the copy, then return it.**

There is a subtlety worth naming. A table has two layers, so there are two different copies:

| Copy | What it duplicates | Protects you from |
|---|---|---|
| `list(rows)` or `rows[:]` | the outer list only | adding or removing **rows** |
| `copy.deepcopy(rows)` | the list *and* every row dictionary | editing a **field inside a row** |

A shallow copy of the list still shares the row dictionaries with the caller, so writing to
`row["price"]` reaches straight back into the original table. The cell below demonstrates
exactly that trap before showing the fix.

In [ ]:
# The coffee-cart table from Session 3, with one missing price.

sales = [
    {"id": 101, "item": "coffee", "price": 3.50, "day": "Mon"},
    {"id": 102, "item": "tea",    "price": None, "day": "Mon"},   # price missing
    {"id": 103, "item": "muffin", "price": 2.50, "day": "Tue"},
]


def surcharged_shallow(rows: list[dict], amount: float) -> list[dict]:
    """Looks safe -- it copies the list. But it still edits the caller's rows."""
    result = list(rows)                      # copies the LIST, not the dictionaries
    for row in result:
        if row["price"] is not None:
            row["price"] += amount           # reaches into the ORIGINAL row
    return result

surcharged_shallow(sales, 0.25)
print("after the shallow copy:", sales[0])   # price is 3.75 -- the raw data changed

A `deepcopy` duplicates the row dictionaries too, so the function can write freely:

In [ ]:
import copy

sales = [
    {"id": 101, "item": "coffee", "price": 3.50, "day": "Mon"},
    {"id": 102, "item": "tea",    "price": None, "day": "Mon"},
    {"id": 103, "item": "muffin", "price": 2.50, "day": "Tue"},
]


def cleaned_sales(rows: list[dict], surcharge: float) -> list[dict]:
    """Return a NEW table of rows that have a price, each with surcharge added.

    Args:
        rows: Sales rows, each with an "id", "item", "price", and "day".
        surcharge: Amount added to every remaining price.

    Returns:
        A new list of new row dictionaries. The supplied rows are not modified.
    """
    result = copy.deepcopy(rows)                                  # both layers
    result = [row for row in result if row["price"] is not None]  # drop missing
    for row in result:
        row["price"] += surcharge
    return result


clean = cleaned_sales(sales, 0.25)

print("returned:")
for row in clean:
    print("  ", row)
print("original row 0:", sales[0])          # still 3.50
print("original rows :", len(sales))        # still 3 -- the None row survives for audit

The copy carries a computational cost, but it protects **provenance** and makes an analysis
pipeline easier to audit: the row with the missing price is still there to be counted and
explained. In performance-sensitive work, document mutation explicitly and test it as part
of the contract.

*This is the same choice pandas gives you in Session 8. `data.copy()` there does the job
`copy.deepcopy` does here, and a pandas operation that warns about a "`SettingWithCopy`"
is warning you about precisely the shallow-copy trap above.*

In [ ]:
### Try it: `add_bonus` below mutates its input. Write TWO versions:
###   add_bonus(records, amount)          -> returns a new list, input unchanged
###   add_bonus_in_place(records, amount) -> returns None, mutates the input
### Then demonstrate the difference by printing the original after each call.

def add_bonus(records, amount):
    for record in records:
        record["pay"] += amount
    return records

staff = [
    {"name": "Ana", "role": "barista", "pay": 100},
    {"name": "Bo",  "role": "cashier", "pay": 120},
]

### Enter your code here ###

---

## 8. Parallel Execution

Parallel or concurrent code makes hidden shared state especially dangerous. If multiple
tasks read and update the same mutable object without coordination, results can depend on
**timing** rather than on the input data. That is a **race condition**.

```python
completed = 0

def record_completion() -> None:
    global completed
    completed += 1
```

If several workers execute this operation concurrently, the final count may not reliably
represent the number of completed tasks. The problem is not merely the global variable's
visibility; it is the **read-modify-write** sequence, which can interleave with another
task:

```text
Thread A reads  completed = 41
Thread B reads  completed = 41
Thread A writes completed = 42
Thread B writes completed = 42     <- one completion vanished
```

The cell below makes the interleaving visible by widening the window between the read and
the write.

In [ ]:
import threading
import time

completed = 0
TASKS_PER_WORKER = 2_000
WORKERS = 4

def record_completion() -> None:
    global completed
    current = completed      # read
    time.sleep(0)            # yield to another thread -- widens the window
    completed = current + 1  # write, using the now-stale value

def worker() -> None:
    for _ in range(TASKS_PER_WORKER):
        record_completion()

threads = [threading.Thread(target=worker) for _ in range(WORKERS)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

expected = TASKS_PER_WORKER * WORKERS
print(f"expected {expected}, got {completed}, lost {expected - completed}")

Run that cell several times. The number changes, and it is usually wrong &mdash; the result
depends on scheduling, not on the data.

> **Do not assume the GIL makes higher-level operations safe.** Python makes limited
> guarantees about the atomicity of operations, and shared mutable data remains a source
> of race conditions.

### The preferred design: workers return results

Prefer a design in which workers receive data, compute independently, and **return**
results. Then aggregate in one controlled location.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def square(value: int) -> int:
    """Pure: explicit input, returned result, no shared state."""
    return value * value

values = list(range(20))

with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(square, values))   # aggregation happens HERE, in one place

print(results)
print("total:", sum(results))
print("deterministic:", results == [v * v for v in values])

Run that cell as many times as you like: the answer is identical every time, because no
worker touches state another worker can see.

### When shared state is unavoidable

Protect **both reads and writes** with appropriate synchronization such as
`threading.Lock`, or use process-safe / shared-data mechanisms suitable for the
concurrency model.

In [ ]:
import threading
import time

completed = 0
lock = threading.Lock()

def record_completion_safe() -> None:
    global completed
    with lock:               # the read-modify-write is now indivisible
        current = completed
        time.sleep(0)        # the SAME yield as before -- the lock still holds
        completed = current + 1

def worker() -> None:
    for _ in range(TASKS_PER_WORKER):
        record_completion_safe()

threads = [threading.Thread(target=worker) for _ in range(WORKERS)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print(f"expected {TASKS_PER_WORKER * WORKERS}, got {completed}")

### Aim for functions close to pure

For parallel data work, the most robust function contracts are often close to **pure
functions**:

- Explicit inputs.
- A returned result.
- No global state.
- No mutation of caller-owned objects.
- No reliance on order or timing of execution.

These properties make functions easier to run with `concurrent.futures`, Dask, Spark UDFs,
workflow engines, test runners, or distributed batch jobs.

Notice that this is the *same* list of properties that made functions easy to test in
Section 3, and easy to audit in Section 7. The design pays off three times.

In [ ]:
### Try it: `tally` below is not safe to run in parallel and its result is not reproducible.
### Rewrite it as a pure function that takes a list of records and RETURNS the counts,
### then run it through ThreadPoolExecutor over several chunks and combine the results.

counts = {}

def tally(row):
    global counts
    counts[row["item"]] = counts.get(row["item"], 0) + 1

cart = [
    {"id": 101, "item": "coffee", "day": "Mon"},
    {"id": 102, "item": "tea",    "day": "Mon"},
    {"id": 103, "item": "coffee", "day": "Tue"},
    {"id": 104, "item": "muffin", "day": "Tue"},
]

### Enter your code here ###

---

## 9. Docstrings and Tests

A docstring records the contract for the next reader &mdash; often the original author after
context has faded.

A useful docstring answers:

- What does this function do?
- What does each parameter mean?
- What types, units, and constraints apply?
- What does it return?
- Does it mutate input or cause external effects?
- Which exceptions or missing-data outcomes should callers expect?

In [ ]:
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from degrees Celsius to degrees Fahrenheit.

    Args:
        celsius: Temperature in degrees Celsius.

    Returns:
        The corresponding temperature in degrees Fahrenheit.
    """
    return celsius * 9 / 5 + 32


help(celsius_to_fahrenheit)

Tests then convert the contract into **executable evidence**:

In [ ]:
def test_celsius_to_fahrenheit():
    assert celsius_to_fahrenheit(0) == 32
    assert celsius_to_fahrenheit(100) == 212
    assert celsius_to_fahrenheit(-40) == -40

test_celsius_to_fahrenheit()
print("test_celsius_to_fahrenheit passed")

For a function with validation, tests should cover **ordinary cases, boundary cases,
invalid types, and invalid values**. `pytest.raises` asserts that the function refuses bad
input &mdash; a test that passes only if the exception *is* raised.

In [ ]:
import pytest


def test_average_valid_input():
    assert average(12.0, 3) == 4.0


def test_average_rejects_zero_count():
    with pytest.raises(ValueError):
        average(12.0, 0)


def test_average_rejects_string_total():
    with pytest.raises(TypeError):
        average("12", 3)


for test in [test_average_valid_input,
             test_average_rejects_zero_count,
             test_average_rejects_string_total]:
    test()
    print(f"{test.__name__} passed")

*If `import pytest` fails in your environment, install it with `%pip install pytest` in a
cell, or replace each `with pytest.raises(...)` block with a `try` / `except` that fails
when no exception is raised.*

Outside a notebook you would put these in a file named `test_*.py` and run `pytest` from a
terminal; it discovers and runs every `test_*` function for you.

### A full contract, fully tested

Below is `validate_age` from Section 5 with the four test categories written out
explicitly.

In [ ]:
def test_validate_age_ordinary():
    assert validate_age("34") == 34
    assert validate_age("28") == 28

def test_validate_age_boundaries():
    assert validate_age("0") == 0        # lower bound is valid
    assert validate_age("120") == 120    # upper bound is valid
    assert validate_age("-1") is None    # just outside
    assert validate_age("121") is None   # just outside

def test_validate_age_invalid_types():
    assert validate_age(None) is None
    assert validate_age([34]) is None

def test_validate_age_invalid_values():
    assert validate_age("") is None
    assert validate_age("not sure") is None

for test in [test_validate_age_ordinary, test_validate_age_boundaries,
             test_validate_age_invalid_types, test_validate_age_invalid_values]:
    test()
    print(f"{test.__name__} passed")

Mutation is part of the contract too, so **test it**:

In [ ]:
def test_without_missing_does_not_mutate():
    original = [1.0, None, 2.0]
    result = without_missing(original)
    assert result == [1.0, 2.0]
    assert original == [1.0, None, 2.0]   # the contract's promise
    assert result is not original


def test_remove_missing_in_place_mutates():
    original = [1.0, None, 2.0]
    returned = remove_missing_in_place(original)
    assert returned is None               # the convention
    assert original == [1.0, 2.0]         # the promised mutation


test_without_missing_does_not_mutate()
test_remove_missing_in_place_mutates()
print("mutation contracts hold")

This is why a clean return value, stable type behavior, explicit edge-case policy, and
limited side effects matter. They turn a claim &mdash; "this function calculates an average"
&mdash; into something that can be **automatically checked each time the code changes**.

In [ ]:
### Try it: write a docstring and four tests (ordinary, boundary, invalid type,
### invalid value) for the `apply_late_fee` function you wrote in Section 2.

### Enter your code here ###

---

## Core Takeaway

> Write functions so that their **names, signatures, docstrings, return values, and tests**
> all tell the same story. When those agree, code becomes easier to reuse, inspect, test,
> parallelize, and trust.

### Checklist for your next function

| Check | Ask yourself |
|---|---|
| **Name** | Does a verb + object name state its one job? |
| **Signature** | Does every input that affects the result appear as a parameter? |
| **Return** | Does it *return* the answer rather than print it? |
| **Types** | Are parameters and the return value annotated? |
| **Edge cases** | Have I decided &mdash; and documented &mdash; what happens for `0`, `None`, negative, empty, and wrong-type input? |
| **Errors** | `TypeError` for the wrong kind of object, `ValueError` for a bad value? |
| **Side effects** | Does it mutate a caller's object or touch the outside world? If so, does the name say so? |
| **Purity** | Could I safely run it many times, or in parallel, and get the same answer? |
| **Docstring** | Does it state args, returns, raises, and any mutation? |
| **Tests** | Is there an ordinary case, a boundary case, an invalid type, and an invalid value? |

### Where to go next

- **Session 5** applies these contracts to file input and output.
- The homework asks you to decompose a working-but-tangled script, exactly like the
  **Try it** cell in Section 5.
- Reference reading: [PEP 257 &mdash; Docstring Conventions](https://peps.python.org/pep-0257/)
  and [PEP 484 &mdash; Type Hints](https://peps.python.org/pep-0484/).